In [0]:
%sql
CREATE CATALOG cdfcatalog;
USE CATALOG cdfcatalog;

In [0]:
%sql
CREATE schema schema1

Before Jumping into Change DataFeed lets see first Ignore Changes

## Ignore Changes

In [0]:
CREATE Table test1.schema1.ignore_test_table
(
  id INT,
  order_name STRING ,
  amount INT,
  prod_id INT
)
USING DELTA;

INSERT INTO test1.schema1.ignore_test_table
VALUES (1,'order1',100,1),(2,'order2',200,2),(3,'order3',300,3);

SELECT * FROM test1.schema1.ignore_test_table

id,order_name,amount,prod_id
1,order1,100,1
2,order2,200,2
3,order3,300,3


In [0]:
%python
from pyspark.sql.functions import current_timestamp
from pyspark.sql.types import StructType, StructField, IntegerType, StringType


schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("order_name", StringType(), True),
    StructField("amount", IntegerType(), True),
    StructField("prod_id", IntegerType(), True)
])

In [0]:
%python

df = (
    spark.readStream
        .format("delta")
        .option("ignoreChanges", "true")
        .schema(schema)
        .table("test1.schema1.ignore_test_table")  # use .table() for catalog table names
)

def print_batch_count(batch_df, batch_id):
    count = batch_df.count()
    
    # log count into a delta table
    spark.createDataFrame(
        [(batch_id, count)], ["batch_id", "row_count"]
    ).write \
        .format("csv") \
        .mode("append") \
        .option("header", "true") \
        .save("abfss://iot@muaazexternalstorage.dfs.core.windows.net/csvfile/")

query = (
    df.writeStream
        .foreachBatch(print_batch_count)
        .option("checkpointLocation", "abfss://iot@muaazexternalstorage.dfs.core.windows.net/checkpoint5/")
        .trigger(processingTime='10 second')  # or processingTime / continuous
        .toTable("test1.schema1.target_table")
)

In [0]:
INSERT INTO test1.schema1.ignore_test_table VALUES (9,'order1',100,1),(10,'order2',200,2),(11,'order3',300,3);

DELETE FROM test1.schema1.ignore_test_table WHERE id=9;

UPDATE test1.schema1.ignore_test_table set amount=2000 WHERE id=9;

num_affected_rows,num_inserted_rows
3,3


In [0]:
%sql
SELECT COUNT(*) FROM test1.schema1.ignore_test_table

count(1)
11


## Change Data Feed

In [0]:
%python
from delta.tables import DeltaTable
from pyspark.sql import functions as F

In [0]:
%sql
-- Enable Changee data feed in New table
CREATE TABLE student 
(
  id INT, 
  name STRING, 
  age INT
)
TBLPROPERTIES (delta.enableChangeDataFeed = true)

In [0]:
%sql
CREATE Table cdfcatalog.schema1.cdf_table
(
  id INT,
  order_name STRING ,
  amount INT,
  prod_id INT
)
USING DELTA;

INSERT INTO cdfcatalog.schema1.cdf_table
VALUES (1,'order1',100,1),(2,'order2',200,2),(3,'order3',300,3);

SELECT * FROM cdfcatalog.schema1.cdf_table

id,order_name,amount,prod_id
1,order1,100,1
2,order2,200,2
3,order3,300,3


In [0]:
-- Enable Change Data Feed on Existing table

ALTER TABLE cdfcatalog.schema1.cdf_table SET TBLPROPERTIES (delta.enableChangeDataFeed = true);

SELECT * FROM cdfcatalog.schema1.cdf_table;

In [0]:
-- Insert data after CDF enabled

INSERT INTO cdfcatalog.schema1.cdf_table
VALUES (4,'order4',100,1),(5,'order5',500,1),(6,'order6',600,2),(7,'order7',700,3);

-- INSERT INTO cdfcatalog.schema1.cdf_table VALUES (8,'order8',800,1),(9,'order9',900,2);
-- INSERT INTO test1.schema1.cdf_table VALUES (9,'order9',900,1),(10,'order10',1000,2),(11,'order11',1100,3);

UPDATE cdfcatalog.schema1.cdf_table SET amount=1000 WHERE id=3;
UPDATE cdfcatalog.schema1.cdf_table SET amount=1200 WHERE id=1;

DELETE FROM cdfcatalog.schema1.cdf_table WHERE id=5;

SELECT * FROM cdfcatalog.schema1.cdf_table


In [0]:
%python
starting_version = 2
ending_version = 5
spark.sql(f"SELECT * FROM table_changes('cdfcatalog.schema1.cdf_table', {starting_version}, {ending_version})") # between 2 versions

spark.sql(f"SELECT * FROM table_changes('cdfcatalog.schema1.cdf_table', {starting_version})") # From starting version till last

In [0]:
%python
# same as above in python
df = (
    spark.read.format("delta")
    .option("readChangeFeed", "true")
    .option("startingVersion", 2)
    .option("endingVersion", 3)
    .table("cdfcatalog.schema1.cdf_table")
)

In [0]:
%python
from pyspark.sql.functions import current_timestamp
from pyspark.sql.types import StructType, StructField, IntegerType, StringType,TimeType,LongType

schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("order_name", StringType(), True),
    StructField("amount", IntegerType(), True),
    StructField("prod_id", IntegerType(), True),
    StructField("_change_type", StringType(), True),
    StructField("_commit_version", LongType(), True),
    StructField("_commit_timestamp", TimeType(), True)
])

In [0]:
%python

from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

window_spec = Window.partitionBy("id") \
    .orderBy(col("_commit_version").desc())

latest_df = (
    df.filter(
        col("_change_type").isin(
            "insert",
            "update_postimage"
        )
    )
    .withColumn(
        "rn",
        row_number().over(window_spec)
    )
    .filter(col("rn") == 1)
    .drop("rn")
)

display(latest_df)

id,order_name,amount,prod_id,_change_type,_commit_version,_commit_timestamp
1,order1,5000,1,update_postimage,12,2026-05-07T07:59:18.000Z
3,order3,1000,3,update_postimage,4,2026-05-05T14:33:22.000Z
4,order4,400,1,insert,3,2026-05-05T14:33:18.000Z
5,order5,500,2,insert,3,2026-05-05T14:33:18.000Z
6,order6,600,3,insert,3,2026-05-05T14:33:18.000Z


In [0]:
%python
Window.partitionBy("id") \
.orderBy(
    col("_commit_version").desc(),
    col("_commit_timestamp").desc()
)

In [0]:
-- Insert Then Update Before Batch Reads

INSERT INTO cdfcatalog.schema1.cdf_table VALUES (10,'order10',9000,1);

UPDATE cdfcatalog.schema1.cdf_table SET amount = 2000 WHERE id=6;

num_affected_rows
1


In [0]:
SELECT * FROM table_changes('cdfcatalog.schema1.cdf_table', 2)

id,order_name,amount,prod_id,_change_type,_commit_version,_commit_timestamp
1,order1,100,1,update_preimage,4,2026-05-05T14:33:22.000Z
1,order1,1000,1,update_postimage,4,2026-05-05T14:33:22.000Z
3,order3,300,3,update_preimage,4,2026-05-05T14:33:22.000Z
3,order3,1000,3,update_postimage,4,2026-05-05T14:33:22.000Z
6,order6,600,3,update_preimage,15,2026-05-07T08:23:33.000Z
6,order6,2000,3,update_postimage,15,2026-05-07T08:23:33.000Z
1,order1,1000,1,update_preimage,8,2026-05-07T07:54:56.000Z
1,order1,3000,1,update_postimage,8,2026-05-07T07:54:56.000Z
1,order1,3000,1,update_preimage,10,2026-05-07T07:59:12.000Z
1,order1,4000,1,update_postimage,10,2026-05-07T07:59:12.000Z


In [0]:
%python
from pyspark.sql.functions import col

df=spark.sql("SELECT * FROM table_changes('cdfcatalog.schema1.cdf_table', 2)")

df = df.filter(
    col("_change_type").isin("insert", "update_postimage")
)

display(df)

id,order_name,amount,prod_id,_change_type,_commit_version,_commit_timestamp
1,order1,1000,1,update_postimage,4,2026-05-05T14:33:22.000Z
3,order3,1000,3,update_postimage,4,2026-05-05T14:33:22.000Z
6,order6,2000,3,update_postimage,15,2026-05-07T08:23:33.000Z
1,order1,3000,1,update_postimage,8,2026-05-07T07:54:56.000Z
1,order1,4000,1,update_postimage,10,2026-05-07T07:59:12.000Z
1,order1,5000,1,update_postimage,12,2026-05-07T07:59:18.000Z
4,order4,400,1,insert,3,2026-05-05T14:33:18.000Z
5,order5,500,2,insert,3,2026-05-05T14:33:18.000Z
6,order6,600,3,insert,3,2026-05-05T14:33:18.000Z
10,order10,9000,1,insert,14,2026-05-07T08:23:29.000Z


In [0]:
-- Delete Scenario
DELETE FROM cdfcatalog.schema1.cdf_table WHERE id=10

In [0]:
%python
delete_df = df.filter(
    col("_change_type") == "delete"
)

display(delete_df)

In [0]:
-- Insert + Delete Before Batch Reads

INSERT INTO cdfcatalog.schema1.cdf_table VALUES (11,'order11',11000,1);

DELETE FROM cdfcatalog.schema1.cdf_table WHERE id=5


In [0]:
-- TRUNCATE TABLE test1.schema1.cdf_sink_table;
TRUNCATE TABLE cdfcatalog.schema1.cdf_table


In [0]:
%python
from pyspark.sql.window import Window

cdf_cols = ["_change_type", "_commit_version", "_commit_timestamp"]

# Define window
window_spec = Window.partitionBy("id").orderBy(F.col("_commit_version").desc())

# Apply filtering + row_number
clean_df = (
    batch_df
    .filter(~F.col("_change_type").isin("update_preimage", "delete"))
    .withColumn("rn", F.row_number().over(window_spec))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

# apply transformation after filtering
clean_df = clean_df.withColumn("amount", F.col("amount") + 10).drop(*cdf_cols)

### Store changes in Permanent table to validate or Evaluate

In [0]:
-- 
CREATE Table cdfcatalog.schema1.test_table
(
  id INT,
  order_name STRING ,
  amount INT,
  prod_id INT,
  _change_type STRING,
  _commit_version LONG,
  _commit_timestamp TIMESTAMP
)
USING DELTA


In [0]:
%python

# Initial Code snippet to validate or evaluate the change data
df=(spark.readStream
  .option("readChangeFeed", "true")
  .option("startingVersion", 3)
  .table("cdfcatalog.schema1.cdf_table")
)

def store_to_test_table(batch_df, batch_id):

    batch_df.write \
    .format("delta") \
    .mode("append") \
    .insertInto("cdfcatalog.schema1.test_table")

    # can perform processsing or transformation before sending to downsttream table 

query = (
    df.writeStream
        .foreachBatch(store_to_test_table)
        .option(
            "checkpointLocation",
            "/Volumes/cdfcatalog/schema1/checkpoints/checkpoint1"
        )
        .trigger(availableNow = True)
        .start()
)


# Documentation Code
(spark.readStream
  .option("readChangeFeed", "true")
  .option("startingVersion", 2) \
  .table("cdfcatalog.schema1.cdf_table")
  .writeStream
  .option("checkpointLocation","/Volumes/cdfcatalog/schema1/checkpoints/checkpoint2")
  .trigger(availableNow=True)
  .toTable("cdfcatalog.schema1.test_table")
)

In [0]:
SELECT * FROM table_changes('cdfcatalog.schema1.cdf_table', 2)

id,order_name,amount,prod_id,_change_type,_commit_version,_commit_timestamp
8,order8,800,1,update_preimage,9,2026-05-08T10:48:52.000Z
8,order8,1200,1,update_postimage,9,2026-05-08T10:48:52.000Z
3,order3,300,3,update_preimage,4,2026-05-08T10:43:14.000Z
3,order3,1000,3,update_postimage,4,2026-05-08T10:43:14.000Z
9,order9,900,2,update_preimage,11,2026-05-08T10:48:57.000Z
9,order9,1500,2,update_postimage,11,2026-05-08T10:48:57.000Z
5,order5,500,1,insert,3,2026-05-08T10:42:17.000Z
6,order6,600,2,insert,3,2026-05-08T10:42:17.000Z
7,order7,700,3,insert,3,2026-05-08T10:42:17.000Z
8,order8,800,1,insert,8,2026-05-08T10:47:55.000Z


In [0]:
%python
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# final validation code to see inserts , deletes and latest updates postimages 

df=spark.sql("SELECT * FROM cdfcatalog.schema1.test_table")

filtered_df = (
    df.filter(
        F.col("_change_type") != "update_preimage"
    )
)

window_spec = Window.partitionBy(
    "id",
    "_commit_version"
).orderBy(
    F.col("_commit_timestamp").desc()
)

final_df = (
    filtered_df
    .withColumn(
        "rn",
        F.when(
            F.col("_change_type") == "update_postimage",
            F.row_number().over(window_spec)
        ).otherwise(1)
    )
    .filter(F.col("rn") == 1)
    .drop("rn")
)

display(final_df)

id,order_name,amount,prod_id,_change_type,_commit_version,_commit_timestamp
3,order3,1000,3,update_postimage,4,2026-05-08T10:43:14.000Z
5,order5,500,1,insert,3,2026-05-08T10:42:17.000Z
5,order5,500,1,delete,6,2026-05-08T10:43:56.000Z
6,order6,600,2,insert,3,2026-05-08T10:42:17.000Z
7,order7,700,3,insert,3,2026-05-08T10:42:17.000Z
8,order8,800,1,insert,8,2026-05-08T10:47:55.000Z
8,order8,1200,1,update_postimage,9,2026-05-08T10:48:52.000Z
9,order9,900,2,insert,8,2026-05-08T10:47:55.000Z
9,order9,1500,2,update_postimage,11,2026-05-08T10:48:57.000Z


In [0]:
%python
# if no primary key

data_cols = [
    c for c in df.columns
    if not c.startswith("_")
]

window_spec = Window.partitionBy(
    *data_cols,
    "_commit_version"
).orderBy(
    F.col("_commit_timestamp").desc()
)

final_df = (
    filtered_df
    .withColumn(
        "rn",
        F.when(
            F.col("_change_type") == "update_postimage",
            F.row_number().over(window_spec)
        ).otherwise(1)
    )
    .filter(F.col("rn") == 1)
    .drop("rn")
)

display(final_df)

id,order_name,amount,prod_id,_change_type,_commit_version,_commit_timestamp
3,order3,1000,3,update_postimage,4,2026-05-08T10:43:14.000Z
5,order5,500,1,insert,3,2026-05-08T10:42:17.000Z
5,order5,500,1,delete,6,2026-05-08T10:43:56.000Z
6,order6,600,2,insert,3,2026-05-08T10:42:17.000Z
7,order7,700,3,insert,3,2026-05-08T10:42:17.000Z
8,order8,800,1,insert,8,2026-05-08T10:47:55.000Z
8,order8,1200,1,update_postimage,9,2026-05-08T10:48:52.000Z
9,order9,900,2,insert,8,2026-05-08T10:47:55.000Z
9,order9,1500,2,update_postimage,11,2026-05-08T10:48:57.000Z


### Multiple Scenarios to see Changes

In [0]:
%python

# to see between specific versions of table 
# startinng_version = 2
# ending_version = 8   
# df=spark.sql("SELECT * FROM table_changes('cdfcatalog.schema1.cdf_table', startinng_version,ending_version)")
# display(df)

# to manually see change data 
df = spark.sql("SELECT * FROM cdfcatalog.schema1.test_table")
display(df)

# to display count of each change type
display(df.groupBy("_change_type").count())

id,order_name,amount,prod_id,_change_type,_commit_version,_commit_timestamp
3,order3,300,3,update_preimage,4,2026-05-08T10:43:14.000Z
3,order3,1000,3,update_postimage,4,2026-05-08T10:43:14.000Z
8,order8,800,1,update_preimage,9,2026-05-08T10:48:52.000Z
8,order8,1200,1,update_postimage,9,2026-05-08T10:48:52.000Z
9,order9,900,2,update_preimage,11,2026-05-08T10:48:57.000Z
9,order9,1500,2,update_postimage,11,2026-05-08T10:48:57.000Z
5,order5,500,1,insert,3,2026-05-08T10:42:17.000Z
6,order6,600,2,insert,3,2026-05-08T10:42:17.000Z
7,order7,700,3,insert,3,2026-05-08T10:42:17.000Z
8,order8,800,1,insert,8,2026-05-08T10:47:55.000Z


_change_type,count
update_preimage,3
update_postimage,3
insert,5
delete,1


In [0]:
%python

# VALIDATE UPDATE PRE/POST PAIRS
# Every updated row should have:
# update_preimage = 1
# update_postimage = 1

from pyspark.sql import functions as F

pair_validation = (
    df
    .filter(
        F.col("_change_type").isin(
            "update_preimage",
            "update_postimage"
        )
    )
    .groupBy("id", "_commit_version")
    .pivot("_change_type")
    .count()
    .fillna(0)
)

display(pair_validation)

id,_commit_version,update_postimage,update_preimage
1,4,1,1
3,4,1,1
6,15,1,1
1,10,1,1
1,12,1,1
1,8,1,1


In [0]:
%python
# Detect rows updated multiple times

multiple_updates = (
    df
    .filter(
        F.col("_change_type") == "update_postimage"
    )
    .groupBy("id")
    .agg(
        F.count("*").alias("update_count")
    )
    .filter(F.col("update_count") > 1)
)

display(multiple_updates)

id,update_count
1,4


In [0]:
%python

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# One final/latest row per id/primary key
# there is fault in it it will only show latest change on primary key

window_spec = Window.partitionBy("id").orderBy(
    F.col("_commit_version").desc(),
    F.col("_commit_timestamp").desc()
)

latest_state = (
    df
    .filter(
        F.col("_change_type").isin(
            "insert",
            "update_postimage",
            "delete"
        )
    )
    .withColumn(
        "rn",
        F.row_number().over(window_spec)
    )
    .filter(F.col("rn") == 1)
    .drop("rn")
)

display(latest_state)

id,order_name,amount,prod_id,_change_type,_commit_version,_commit_timestamp
3,order3,1000,3,update_postimage,4,2026-05-08T10:43:14.000Z
5,order5,500,1,delete,6,2026-05-08T10:43:56.000Z
6,order6,600,2,insert,3,2026-05-08T10:42:17.000Z
7,order7,700,3,insert,3,2026-05-08T10:42:17.000Z
8,order8,1200,1,update_postimage,9,2026-05-08T10:48:52.000Z
9,order9,1500,2,update_postimage,11,2026-05-08T10:48:57.000Z


In [0]:
%python

# to see All deleted rows
delete_df = (
    df
    .filter(
        F.col("_change_type") == "delete"
    )
)

display(delete_df)

id,order_name,amount,prod_id,_change_type,_commit_version,_commit_timestamp
5,order5,500,2,delete,6,2026-05-05T14:33:25.000Z


In [0]:
%python
# see version timestamp
version_df = (
    df
    .select(
        "_commit_version",
        "_commit_timestamp"
    )
    .distinct()
    .orderBy("_commit_version")
)

display(version_df)

_commit_version,_commit_timestamp
3,2026-05-05T14:33:18.000Z
4,2026-05-05T14:33:22.000Z
6,2026-05-05T14:33:25.000Z
8,2026-05-07T07:54:56.000Z
10,2026-05-07T07:59:12.000Z
12,2026-05-07T07:59:18.000Z
14,2026-05-07T08:23:29.000Z
15,2026-05-07T08:23:33.000Z


### Send changes to downstream table in Streaming 

In [0]:
%python
from pyspark.sql.window import Window

def merge_to_sink(batch_df, batch_id):

    cdf_cols = ["_change_type", "_commit_version", "_commit_timestamp"]

    # Define window
    window_spec = Window.partitionBy("id").orderBy(F.col("_commit_version").desc())

    # Apply filtering + row_number
    clean_df = (
        batch_df
        .filter(~F.col("_change_type").isin("update_preimage", "delete"))
        .withColumn("rn", F.row_number().over(window_spec))
        .filter(F.col("rn") == 1)
        .drop("rn")
    )

    # apply transformation after filtering
    clean_df = clean_df.withColumn("amount", F.col("amount") + 10).drop(*cdf_cols)

    clean_df.write \
    .format("delta") \
    .mode("append") \
    .insertInto("cdfcatalog.schema1.cdf_sink_table") # store data in target tables 



%python
query = (
    df.writeStream
        .foreachBatch(merge_to_sink)
        .option(
            "checkpointLocation",
            "/Volumes/cdfcatalog/schema1/checkpoints/checkpoint3"
        )
        .trigger(availableNow = True)
        .start()
)


In [0]:
SELECT * FROM cdfcatalog.schema1.cdf_sink_table;

In [0]:
DESCRIBE HISTORY cdfcatalog.schema1.cdf_table

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
5,2026-04-29T13:27:44Z,143405734051923,muaazmuzammil69@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2315695601999904),0428-101353-istrad9e,4,SnapshotIsolation,false,"Map(numRemovedFiles -> 3, numRemovedBytes -> 4930, p25FileSize -> 1621, numDeletionVectorsRemoved -> 1, minFileSize -> 1621, numAddedFiles -> 1, maxFileSize -> 1621, p75FileSize -> 1621, p50FileSize -> 1621, numAddedBytes -> 1621)",null,Databricks-Runtime/17.3.x-scala2.13
4,2026-04-29T13:27:39Z,143405734051923,muaazmuzammil69@gmail.com,UPDATE,"Map(predicate -> [""(id#13934 = 1)""])",null,List(2315695601999904),0428-101353-istrad9e,3,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 1, executionTimeMs -> 3407, numDeletionVectorsUpdated -> 0, scanTimeMs -> 1521, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 1940, rewriteTimeMs -> 1885)",null,Databricks-Runtime/17.3.x-scala2.13
3,2026-04-29T13:27:18Z,143405734051923,muaazmuzammil69@gmail.com,WRITE,"Map(mode -> Append, partitionBy -> [], statsOnLoad -> false)",null,List(2315695601999904),0428-101353-istrad9e,2,WriteSerializable,true,"Map(numFiles -> 1, numOutputBytes -> 1495, numOutputRows -> 3)",null,Databricks-Runtime/17.3.x-scala2.13
2,2026-04-29T13:26:22Z,143405734051923,muaazmuzammil69@gmail.com,SET TBLPROPERTIES,"Map(properties -> {""delta.enableChangeDataFeed"":""true""})",null,List(2315695601999904),0428-101353-istrad9e,1,WriteSerializable,true,Map(),null,Databricks-Runtime/17.3.x-scala2.13
1,2026-04-29T13:26:15Z,143405734051923,muaazmuzammil69@gmail.com,WRITE,"Map(mode -> Append, partitionBy -> [], statsOnLoad -> false)",null,List(2315695601999904),0428-101353-istrad9e,0,WriteSerializable,true,"Map(numFiles -> 1, numOutputBytes -> 1495, numOutputRows -> 3)",null,Databricks-Runtime/17.3.x-scala2.13
0,2026-04-29T13:26:13Z,143405734051923,muaazmuzammil69@gmail.com,CREATE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> false, properties -> {""delta.enableDeletionVectors"":""true""}, statsOnLoad -> false)",null,List(2315695601999904),0428-101353-istrad9e,null,WriteSerializable,true,Map(),null,Databricks-Runtime/17.3.x-scala2.13


In [0]:
DESCRIBE HISTORY cdfcatalog.schema1.cdf_sink_table

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
4,2026-04-29T09:27:32Z,143405734051923,muaazmuzammil69@gmail.com,WRITE,"Map(mode -> Append, partitionBy -> [], statsOnLoad -> false)",null,List(2315695601999904),0428-101353-istrad9e,3,WriteSerializable,true,"Map(numFiles -> 0, numOutputBytes -> 0, numOutputRows -> 0)",null,Databricks-Runtime/17.3.x-scala2.13
3,2026-04-29T09:27:22Z,143405734051923,muaazmuzammil69@gmail.com,WRITE,"Map(mode -> Append, partitionBy -> [], statsOnLoad -> false)",null,List(2315695601999904),0428-101353-istrad9e,2,WriteSerializable,true,"Map(numFiles -> 1, numOutputBytes -> 1546, numOutputRows -> 2)",null,Databricks-Runtime/17.3.x-scala2.13
2,2026-04-29T09:26:42Z,143405734051923,muaazmuzammil69@gmail.com,WRITE,"Map(mode -> Append, partitionBy -> [], statsOnLoad -> false)",null,List(2315695601999904),0428-101353-istrad9e,1,WriteSerializable,true,"Map(numFiles -> 1, numOutputBytes -> 1465, numOutputRows -> 3)",null,Databricks-Runtime/17.3.x-scala2.13
1,2026-04-29T09:25:40Z,143405734051923,muaazmuzammil69@gmail.com,WRITE,"Map(mode -> Append, partitionBy -> [], statsOnLoad -> false)",null,List(2315695601999904),0428-101353-istrad9e,0,WriteSerializable,true,"Map(numFiles -> 1, numOutputBytes -> 1465, numOutputRows -> 3)",null,Databricks-Runtime/17.3.x-scala2.13
0,2026-04-29T09:13:17Z,143405734051923,muaazmuzammil69@gmail.com,CREATE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> false, properties -> {""delta.enableDeletionVectors"":""true""}, statsOnLoad -> false)",null,List(2315695601999904),0429-061652-c2o93h9d-v2n,null,WriteSerializable,true,Map(),null,Databricks-Runtime/18.1.x-photon-scala2.13


## Change Data Feed via S1 and S2

In [0]:
CREATE TABLE student_source_scd(
    student_id INT PRIMARY KEY,
    name       VARCHAR(100),
    dept       VARCHAR(50)
);

In [0]:
-- INSERT INTO student_source_scd VALUES (1,'John','Maths');
-- INSERT INTO student_source_scd VALUES (2,'Mary','Science');
-- INSERT INTO student_source_scd VALUES (3,'Peter','English');

-- UPDATE student_source_scd SET dept = 'Computer' WHERE student_id = 2;

DELETE FROM student_source_scd WHERE student_id = 3


num_affected_rows
1


In [0]:
CREATE TABLE student_scd1 (
    student_id INT PRIMARY KEY,
    name       VARCHAR(100),
    dept       VARCHAR(50)
);

In [0]:
MERGE INTO student_scd1 as target
USING student_source_scd as source
ON target.student_id = source.student_id
WHEN MATCHED AND (
    target.name <> source.name OR target.dept <> source.dept
) THEN
UPDATE SET target.name = source.name, target.dept = source.dept
WHEN NOT MATCHED THEN
INSERT (student_id,name,dept) VALUES (source.student_id, source.name, source.dept)
WHEN NOT MATCHED BY SOURCE THEN DELETE;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
1,0,1,0


In [0]:
SELECT * FROM student_scd1

student_id,name,dept
1,John,Maths
2,Mary,Computer


In [0]:
CREATE TABLE student_scd2 (
    student_id   INT,
    name         VARCHAR(100),
    dept         VARCHAR(50),

    start_date   DATE,
    end_date     DATE,
    is_current   BOOLEAN
);

In [0]:
-- INSERT INTO student_source_scd(student_id,name,dept)
-- VALUES (5,'John','Algo'), (6,'Mary','DSA'), (7,'Peter','Assembly');

UPDATE student_source_scd SET dept = 'Computer' WHERE student_id = 6

-- DELETE FROM student_source_scd WHERE student_id =5


num_affected_rows
1


In [0]:
MERGE INTO student_scd2 AS target
USING student_source_scd AS source
ON target.student_id = source.student_id AND target.is_current = true

WHEN MATCHED AND (
       target.duck_name <> source.duck_name OR
       target.breed     <> source.breed     OR
       target.location  <> source.location
) THEN UPDATE SET
    end_date    = CURRENT_DATE - INTERVAL '1 day',
    is_current  = false

WHEN NOT MATCHED BY SOURCE AND target.is_current = true THEN UPDATE SET
    end_date    = CURRENT_DATE - INTERVAL '1 day',
    is_current  = false

WHEN NOT MATCHED BY TARGET THEN INSERT (
    record_id, duck_id, duck_name, breed, location,
    begin_date, end_date, is_current
) VALUES (
    nextval('duck_record_seq'),
    source.duck_id, source.duck_name, source.breed, source.location,
    source.begin_date, source.end_date, source.is_current
)

RETURNING merge_action, *;

---------------------------------------------------------------------------
ParseException                            Traceback (most recent call last)
File <command-6329863888934329>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', "INSERT INTO student_scd2 (student_id,name,dept,start_date,end_date,is_current)\nSELECT student_id,name,dept,start_date,end_date,is_current\nFROM (\n    MERGE INTO student_scd2 AS target\n    USING student_source_scd AS source\n    ON target.student_id = source.student_id AND target.is_current = true\n    WHEN MATCHED THEN\n        UPDATE SET \n            target.is_current = false,\n            target.end_date = current_date\n    WHEN NOT MATCHED THEN\n        INSERT (student_id, name, salary, start_date, end_date, is_current)\n        VALUES (source.student_id, source.name, source.salary, current_date, NULL, true)\n    OUTPUT ,\n        source.student_id,\n        source.name,\n        source.salary,\n        current_date,\n        NULL,\n        t

In [0]:
MERGE INTO student_scd2 as target
USING student_source_scd as source
ON target.student_id = source.student_id AND target.is_current = true
WHEN MATCHED AND (
    target.name <> source.name OR target.dept <> source.dept
) THEN
UPDATE SET target.end_date = current_date , is_current = false;

INSERT INTO student_scd2 SELECT source.student_id, source.name,
WHEN NOT MATCHED THEN
INSERT (student_id,name,dept,start_date,end_date,is_current) VALUES (source.student_id, source.name, source.dept, current_date, null, true)


---------------------------------------------------------------------------
ParseException                            Traceback (most recent call last)
File <command-7260502959024381>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'MERGE INTO student_scd2 as target\nUSING student_source_scd as source\nON target.student_id = source.student_id\nWHEN MATCHED AND (\n    target.name <> source.name OR target.dept <> source.dept\n) THEN\nUPDATE SET target.name = source.name, target.dept = source.dept , target.end_date = current_date , is_current = false\nINSERT (student_id,name,dept,start_date,end_date,is_current) VALUES (source.student_id, source.name, source.dept, current_date,null,true)\nWHEN NOT MATCHED THEN\nINSERT (student_id,name,dept,start_date,end_date,is_current) VALUES (source.student_id, source.name, source.dept, current_date, null, true)\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(sel

In [0]:
SELECT * FROM student_scd2

student_id,name,dept,start_date,end_date,is_current
6,Mary,Computer,2026-05-12,null,false
1,John,Maths,2026-05-12,null,true
2,Mary,Computer,2026-05-12,null,true
5,John,Algo,2026-05-12,null,true
7,Peter,Assembly,2026-05-12,null,true


num_affected_rows
1
